# Week 5 — Demand Signals & Customer-Behavior Analysis

Builds a comparable **demand-signal index** across the five InGen verticals from three public signal families — Google Trends search interest, GDELT news cadence/sentiment, and voice-of-customer review mining — and surfaces a pain-point taxonomy per vertical.

Real APIs are wired in each collector; where network is unavailable the pipeline uses clearly-labelled schema-correct samples so it runs end-to-end (set `INGEST_ALLOW_NETWORK=1` for live data).

## 1. Run the pipeline + reports

In [ ]:
import subprocess, sys
for mod in ['src.signals.run_all','src.signals.build_reports']:
    print('running', mod)
    print(subprocess.run([sys.executable,'-m',mod],capture_output=True,text=True,cwd='..').stdout[-600:])

## 2. Demand-signal index — which vertical is heating up now

In [ ]:
import pandas as pd
idx = pd.read_csv('../data/week05/demand_signal_index.csv')
print(idx[['rank','product','demand_index','search_score','news_vol_score','news_tone_score']].to_string(index=False))

## 3. Index composition (weighted contributions)

In [ ]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from src.signals.demand_index import WEIGHTS
c = idx.sort_values('demand_index')
left=[0]*len(c)
for col,w,color,lab in (('search_score',WEIGHTS['search_momentum'],'#2E5496','Search'),
                        ('news_vol_score',WEIGHTS['news_momentum'],'#8FAADC','News vol'),
                        ('news_tone_score',WEIGHTS['news_sentiment'],'#C6D2E8','Sentiment')):
    contrib=(c[col]*w).tolist(); plt.barh(c['product'],contrib,left=left,color=color,label=lab)
    left=[l+x for l,x in zip(left,contrib)]
plt.legend(); plt.xlabel('weighted contribution'); plt.title('Demand-index composition'); plt.tight_layout()
plt.savefig('week05_index_composition.png',dpi=110); print('saved')

## 4. Search-interest trends

In [ ]:
si = pd.read_csv('../data/week05/search_interest_long.csv')
for v in si['vertical'].unique():
    s=si[si.vertical==v].sort_values('date')
    plt.plot(range(len(s)), s['value'], label=s['product'].iloc[0])
plt.legend(); plt.xlabel('months (old->new)'); plt.ylabel('interest (0-100)'); plt.title('Search interest'); plt.tight_layout()
plt.savefig('week05_search_trends.png',dpi=110); print('saved')

## 5. Voice-of-customer pain points

In [ ]:
pp = pd.read_csv('../data/week05/pain_points_long.csv')
for v in pp['vertical'].unique():
    sub=pp[pp.vertical==v].sort_values('size',ascending=False)
    print(f"\n=== {sub['product'].iloc[0]} ({v}) ===")
    for _,r in sub.iterrows():
        print(f"  - ({r['size']}) {r['top_terms']}")

## 6. Takeaways for InGen
- The **demand-signal index** ranks verticals by *current momentum* (search + news + sentiment), complementing Week 4's *market size*. A vertical can be large but cooling, or small but heating up — leadership should read the two together.
- **Pain-point taxonomy** gives product/marketing concrete messaging angles per vertical (battery life, navigation/getting stuck, setup complexity, price/ROI recur across verticals).
- All figures are **relative public-signal reads, not forecasts**; validate against internal pipeline/CRM before planning. Run mode (live vs sample) is recorded per source in `data/week05/signal_manifest.jsonl`.